# Notebook 01: Market Data Exploration

**Purpose**: validate market data quality and coverage before calibration.

This notebook:
1. loads a representative SPY option-chain snapshot from the repo's archived data,
2. applies the same style of liquidity and sanity filters used by the canonical workflow,
3. visualizes strike-maturity coverage,
4. examines implied-volatility structure,
5. and identifies which part of the surface is liquid enough to trust downstream.


### Why This Notebook Comes First
Before building any model, we need to answer a simple question: do we trust the market data enough to differentiate it later? Dupire local volatility is very sensitive to noise, so this notebook is deliberately conservative. The goal is not to price anything yet. The goal is to understand what the raw option chain looks like, what must be filtered out, and what coverage remains after cleaning.


In [ ]:
import os
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src directory to path so modules are importable at top level
src_path = Path.cwd().parent / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Added to path: {src_path}")

warnings.filterwarnings("ignore")

from market_data.loaders import load_any
from market_data.validators import validate_and_clean, ValidationConfig
from market_data.transforms import add_derived_columns, TransformConfig

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
os.makedirs('../output/figures', exist_ok=True)

print("✓ Imports successful")


## 1. Load Market Data

The next cell loads a single archived option-chain snapshot and records the basic market context: spot, maturities, strikes, and quote fields. In the current repo, the canonical daily collector archives Theta option snapshots at 15:45 ET and appends them into a dated panel. This notebook uses one archived snapshot from that workflow, or falls back to the older local sample snapshot if no archive is available.


In [ ]:
# === Market data source ===
TICKER = "SPY"
PROJECT_ROOT = Path.cwd().parent
ARCHIVE_ROOT = PROJECT_ROOT / "data" / "archive" / TICKER
SAMPLE_FALLBACK = PROJECT_ROOT / "data" / "raw" / "spy_options_snapshot.parquet"

snapshot_candidates = sorted(ARCHIVE_ROOT.glob("*/option_chain_snapshot.parquet")) if ARCHIVE_ROOT.exists() else []
SNAPSHOT_PATH = snapshot_candidates[-1] if snapshot_candidates else SAMPLE_FALLBACK

print(f"Loading snapshot from: {SNAPSHOT_PATH}")
df_raw = load_any(SNAPSHOT_PATH)

spot_price = float(pd.to_numeric(df_raw["underlying_price"], errors="coerce").dropna().iloc[0])
print(f"? Loaded {len(df_raw)} contracts")
print(f"  Spot: ${spot_price:.2f}")
print(f"  Columns: {list(df_raw.columns)}")
print(f"  Date: {pd.to_datetime(df_raw['date']).iloc[0].date()}")


## 2. Validation & Cleaning

Cleaning matters because every downstream derivative, density, and local-vol estimate amplifies bad quotes. The validation rules below mirror the current canonical liquid-core philosophy of the repo: keep the most usable part of the listed chain and do not pretend the full option universe is equally trustworthy.


In [ ]:
# Configure validation (aligned with the current canonical liquid-core workflow)
val_cfg = ValidationConfig(
    min_volume=25,
    min_open_interest=200,
    enforce_bid_ask=True,
    allow_negative_mid=False,
)

# Validate and clean
df_clean, report = validate_and_clean(df_raw, cfg=val_cfg)

print("Validation Results:")
print(f"  Input: {report['n_in']} contracts")
print(f"  Output: {report['n_out']} contracts")
print(f"  Dropped: {report['dropped']} ({100*report['dropped']/report['n_in']:.1f}%)")
print(f"
Drop reasons:")
for reason, count in report['drop_counts'].items():
    if count > 0:
        print(f"  {reason}: {count}")

processed_dir = Path('../data/processed')
processed_dir.mkdir(exist_ok=True)
df_clean.to_csv(processed_dir / 'cleaned_options_data.csv', index=False)
print(f"
? Saved cleaned data to {processed_dir / 'cleaned_options_data.csv'}")


## 3. Add Derived Columns

Once the quote set is cleaned, the notebook enriches it with the coordinates used throughout the rest of the project. Forward price, moneyness, log-moneyness, and time-to-expiry are the natural language of surface construction, so we compute them here and keep them for all later stages.


In [ ]:
# Configure transforms
trans_cfg = TransformConfig(
    day_count="ACT/365",
    add_forward=True,
    add_log_moneyness=True,
    compute_time_to_expiry_if_missing=True
)

# Add derived columns
df = add_derived_columns(df_clean, cfg=trans_cfg)

# Add spreads
df['bid_ask_spread'] = df['ask'] - df['bid']
df['relative_spread'] = df['bid_ask_spread'] / df['mid']

# Add moneyness (K/S)
df['moneyness'] = df['strike'] / df['underlying_price']

# Add days to expiry for plotting
df['days_to_expiry'] = df['time_to_expiry'] * 365

print("✓ Derived columns added")
print(f"  New columns: forward, log_moneyness, moneyness, days_to_expiry")


In [ ]:
n_before = len(df)

df = df[
    (df['moneyness'] >= 0.90) &
    (df['moneyness'] <= 1.10)
].copy()

df = df[df['relative_spread'] <= 0.05].copy()

n_after = len(df)
print(f"?? Moneyness + Spread Filtering:")
print(f"   Before: {n_before} options")
print(f"   After: {n_after} options ({100*n_after/n_before:.1f}%)")
print(f"   Moneyness range: [{df['moneyness'].min():.2f}, {df['moneyness'].max():.2f}]")
print(f"   Avg relative spread: {df['relative_spread'].mean():.3f}")


In [ ]:
from arbitrage.static_checks import check_static_arbitrage_calls

violations = []
for T in sorted(df['time_to_expiry'].unique()):
    T_df = df[df['time_to_expiry'] == T].sort_values('strike')
    
    if len(T_df) >= 3:
        # Check butterfly spreads (C(K-) - 2*C(K) + C(K+) >= 0)
        K = T_df['strike'].values
        C = T_df['mid'].values  # Use mid prices
        
        for i in range(1, len(K)-1):
            dK1 = K[i] - K[i-1]
            dK2 = K[i+1] - K[i]
            
            # Normalized butterfly
            butterfly = (C[i-1] - C[i])/dK1 - (C[i] - C[i+1])/dK2
            
            if butterfly < -1e-6:  # Negative = arbitrage
                violations.append({
                    'T': T,
                    'K_mid': K[i],
                    'butterfly': butterfly
                })

print(f"🦋 Butterfly Arbitrage Check:")
print(f"   Total violations: {len(violations)}")
if len(violations) > 0:
    print(f"   Worst violation: {min(v['butterfly'] for v in violations):.6f}")
    print(f"   Affected maturities: {len(set(v['T'] for v in violations))}")
else:
    print(f"   ✅ No violations detected!")


## 4. Visualizations

### 4.1 Bid-Ask Spreads

The visualizations below answer three practical questions. First, where does the data actually live in strike-maturity space? Second, what parts of the surface are dense or sparse? Third, after filtering, do we still have enough structure left to support an implied-volatility and local-volatility workflow?


In [ ]:
df.info()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute
axes[0].hist(df['bid_ask_spread'], bins=50, edgecolor='k', alpha=0.7)
p95 = df['bid_ask_spread'].quantile(0.95)
axes[0].axvline(p95, color='red', linestyle='--', linewidth=2, label=f'95%: ${p95:.2f}')
axes[0].set_xlabel('Bid-Ask Spread ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Absolute Spread')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Relative
rel_capped = np.clip(df['relative_spread'], 0, 0.5)
axes[1].hist(rel_capped, bins=50, edgecolor='k', alpha=0.7, color='coral')
p95_rel = df['relative_spread'].quantile(0.95)
axes[1].axvline(p95_rel, color='red', linestyle='--', linewidth=2, label=f'95%: {p95_rel:.1%}')
axes[1].set_xlabel('Relative Spread')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Relative Spread')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../output/figures/01_spreads.png', dpi=150)
plt.show()

print(f"Mean: ${df['bid_ask_spread'].mean():.3f} ({df['relative_spread'].mean():.2%})")


### 4.2 Coverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Volume
sc1 = axes[0].scatter(df['log_moneyness'], df['days_to_expiry'],
                      c=np.log10(df['volume']+1), s=30, alpha=0.6,
                      cmap='viridis', edgecolors='k', linewidth=0.3)
axes[0].axvline(0, color='red', linestyle='--', alpha=0.5, label='ATM')
axes[0].set_xlabel('Log-Moneyness ln(K/S)')
axes[0].set_ylabel('Days to Expiry')
axes[0].set_title('Coverage (Volume)')
axes[0].legend()
axes[0].grid(alpha=0.3)
plt.colorbar(sc1, ax=axes[0], label='log₁₀(Volume)')

# Open Interest
sc2 = axes[1].scatter(df['log_moneyness'], df['days_to_expiry'],
                      c=np.log10(df['open_interest']+1), s=30, alpha=0.6,
                      cmap='plasma', edgecolors='k', linewidth=0.3)
axes[1].axvline(0, color='red', linestyle='--', alpha=0.5, label='ATM')
axes[1].set_xlabel('Log-Moneyness ln(K/S)')
axes[1].set_ylabel('Days to Expiry')
axes[1].set_title('Coverage (Open Interest)')
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.colorbar(sc2, ax=axes[1], label='log₁₀(OI)')

plt.tight_layout()
plt.savefig('../output/figures/01_coverage.png', dpi=150)
plt.show()

print(f"Log-moneyness: [{df['log_moneyness'].min():.3f}, {df['log_moneyness'].max():.3f}]")
print(f"DTE: [{df['days_to_expiry'].min():.0f}, {df['days_to_expiry'].max():.0f}]")
print(f"Expirations: {df['maturity_date'].nunique()}")



### 4.3 Implied Volatility (if available)

In [ ]:
# Check if IV is in the data
if 'impliedVolatility' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Raw scatter
    sc = axes[0,0].scatter(df['log_moneyness'], df['impliedVolatility'],
                           c=df['days_to_expiry'], s=20, alpha=0.5, cmap='coolwarm')
    axes[0,0].axvline(0, color='red', linestyle='--', alpha=0.5)
    axes[0,0].set_xlabel('Log-Moneyness')
    axes[0,0].set_ylabel('Implied Volatility')
    axes[0,0].set_title('IV Surface')
    axes[0,0].grid(alpha=0.3)
    plt.colorbar(sc, ax=axes[0,0], label='DTE')
    
    # By maturity buckets
    df['dte_bucket'] = pd.cut(df['days_to_expiry'],
                               bins=[0, 30, 60, 90, 180, 365, 1000],
                               labels=['0-30', '30-60', '60-90', '90-180', '180-365', '365+'])
    
    for bucket in ['0-30', '30-60', '60-90', '90-180']:
        subset = df[df['dte_bucket'] == bucket]
        if len(subset) > 0:
            axes[0,1].scatter(subset['log_moneyness'], subset['impliedVolatility'],
                             label=f'{bucket} DTE', alpha=0.6, s=15)
    
    axes[0,1].axvline(0, color='red', linestyle='--', alpha=0.5)
    axes[0,1].set_xlabel('Log-Moneyness')
    axes[0,1].set_ylabel('Implied Volatility')
    axes[0,1].set_title('IV by Maturity')
    axes[0,1].legend()
    axes[0,1].grid(alpha=0.3)
    
    # ATM term structure
    atm = df[(df['moneyness'] > 0.98) & (df['moneyness'] < 1.02)]
    atm_grouped = atm.groupby('days_to_expiry')['impliedVolatility'].mean().reset_index()
    
    axes[1,0].plot(atm_grouped['days_to_expiry'], atm_grouped['impliedVolatility'],
                   'o-', markersize=6, linewidth=2, color='navy')
    axes[1,0].set_xlabel('Days to Expiry')
    axes[1,0].set_ylabel('ATM IV')
    axes[1,0].set_title('ATM Term Structure')
    axes[1,0].grid(alpha=0.3)
    
    # Liquidity
    axes[1,1].scatter(np.log10(df['volume']+1), np.log10(df['open_interest']+1),
                      c=df['relative_spread'], s=20, alpha=0.5, cmap='RdYlGn_r')
    axes[1,1].set_xlabel('log₁₀(Volume)')
    axes[1,1].set_ylabel('log₁₀(OI)')
    axes[1,1].set_title('Liquidity')
    axes[1,1].grid(alpha=0.3)
    plt.colorbar(axes[1,1].collections[0], ax=axes[1,1], label='Rel. Spread')
    
    plt.tight_layout()
    plt.savefig('../output/figures/01_iv_structure.png', dpi=150)
    plt.show()
else:
    print("No impliedVolatility column in data")

### 4.4 Filter Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Volume
axes[0].hist(np.log10(df_raw['volume']+1), bins=50, alpha=0.5, label='Before', edgecolor='k')
axes[0].hist(np.log10(df['volume']+1), bins=50, alpha=0.7, label='After', edgecolor='k')
axes[0].axvline(np.log10(11), color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('log₁₀(Volume+1)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Volume Filter (min=25)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# OI
axes[1].hist(np.log10(df_raw['open_interest']+1), bins=50, alpha=0.5, label='Before', edgecolor='k')
axes[1].hist(np.log10(df['open_interest']+1), bins=50, alpha=0.7, label='After', edgecolor='k')
axes[1].axvline(np.log10(6), color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('log₁₀(OI+1)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Open Interest Filter (min=200)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../output/figures/01_filters.png', dpi=150)
plt.show()

## 5. Summary

This summary closes the data stage. By the end of this notebook, the reader should know what data survived cleaning, which regions of the market are well covered, and why the cleaned dataset is a reasonable starting point for a surface model.


In [ ]:
print("="*70)
print("FINAL DATASET")
print("="*70)
print(f"Contracts: {len(df)}")
print(f"Spot: ${spot_price:.2f}")
print(f"Expirations: {df['maturity_date'].nunique()}")
print(f"Date range: {df['maturity_date'].min()} to {df['maturity_date'].max()}")
print()
df[['strike', 'mid', 'volume', 'open_interest',
    'moneyness', 'days_to_expiry', 'relative_spread']].describe()

In [ ]:
print("\nMATURITY BUCKETS")
df['dte_bucket'] = pd.cut(df['days_to_expiry'],
                           bins=[0, 30, 60, 90, 180, 365, 1000],
                           labels=['0-30', '30-60', '60-90', '90-180', '180-365', '365+'])
print(df['dte_bucket'].value_counts().sort_index())


In [ ]:
print("\nMONEYNESS BUCKETS")
df['money_bucket'] = pd.cut(df['moneyness'],
                             bins=[0, 0.9, 0.95, 0.98, 1.02, 1.05, 1.1, 2.0],
                             labels=['Deep OTM', 'OTM', 'Slight OTM', 'ATM',
                                    'Slight ITM', 'ITM', 'Deep ITM'])
print(df['money_bucket'].value_counts().sort_index())

In [ ]:
output_path = '../output/cleaned_options_data.csv'
df.to_csv(output_path, index=False)
print(f"✓ Saved: {output_path}")
print(f"  {len(df)} rows × {len(df.columns)} columns")